# 📧 Email Spam Detection — Multilingual (EN + AR)
**Intelligent Programming Project**

Classify messages as **Spam** or **Ham** using Machine Learning.
Supports **English 🇬🇧 and Arabic 🇸🇦** with Hybrid Detection.

---

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from arabic_data import get_all_arabic_data

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)

print('✅ Libraries imported successfully!')

## 2️⃣ Spam Keywords (EN + AR)

In [ ]:
EN_SPAM_KW = [
    'winner','won','prize','free','claim','urgent','congratulations','click here',
    'limited offer','cash','lottery','selected','reward','guaranteed','exclusive',
    'discount','act now','call now','order now','sign up','verify','suspended',
    'account suspended','credit','loan','investment','earn money','make money',
    'work from home','no experience','risk free','casino','jackpot','bitcoin',
    'double your','bank details','personal details','buy now','get rich','secret',
    'apply now','special offer','you have been selected','you have won',
]

AR_SPAM_KW = [
    'مبروك','فزت','جائزه','جوائز','مجانا','مجاني','خصم','عرض خاص','اتصل الان',
    'سارع','محدود','حصري','ربح','اكسب','استثمر','قرض','تمويل','بدون فوائد',
    'موافقه','تهانينا','تم اختيارك','عاجل','فوري','تحقق','بياناتك','حسابك',
    'اختراق','ايقاف','تجديد','اشتراكك','انتهي','انقر هنا','اضغط هنا',
    'سجل الان','اشترك الان','هديه مجانيه','رحله مجانيه','ايفون مجاني',
    'لابتوب مجاني','ضاعف اموالك','ارسل بياناتك','رقم حسابك','تسوق الان',
    'اطلب الان','توصيل مجاني','ادويه رخيصه','ربح سريع','فرصه ذهبيه',
    'استثمار مضمون','عائد مضمون','كسب من المنزل','بدون خبره','تجميد حسابك',
    'نشاط مشبوه','تحديث عاجل','رابط تسجيل','انتهت صلاحيته','واتساب',
    'تلجرام','عضويه مجانيه','مليونير','الثروه','دخل سلبي','تداول','فوركس',
]

def keyword_check(text):
    """Check spam keywords. Returns (is_spam, matched_keyword)."""
    t_low = text.lower()
    t_norm = re.sub(r'[أإآا]', 'ا', t_low)
    t_norm = re.sub(r'ة', 'ه', t_norm)
    t_norm = re.sub(r'ى', 'ي', t_norm)
    t_norm = re.sub(r'[\u064B-\u065F\u0670]', '', t_norm)
    for kw in EN_SPAM_KW:
        if kw in t_low:
            return True, kw
    for kw in AR_SPAM_KW:
        kw_n = re.sub(r'[أإآا]','ا',kw)
        kw_n = re.sub(r'ة','ه',kw_n)
        kw_n = re.sub(r'ى','ي',kw_n)
        if kw_n in t_norm:
            return True, kw
    return False, None

print(f'✅ EN keywords: {len(EN_SPAM_KW)} | AR keywords: {len(AR_SPAM_KW)}')

## 3️⃣ Load Dataset (English + Arabic)

In [ ]:
# English dataset
df_en = pd.read_csv('spam.csv', encoding='latin-1')
df_en = df_en[['v1', 'v2']]
df_en.columns = ['label', 'message']
print(f'English dataset: {len(df_en)} messages')

# Arabic dataset (314 messages across 4 spam categories + ham)
ar_data = get_all_arabic_data()
df_ar = pd.DataFrame(ar_data, columns=['message', 'label'])
print(f'Arabic dataset : {len(df_ar)} messages')
print(f'  Spam: {sum(1 for _,l in ar_data if l=="spam")} | Ham: {sum(1 for _,l in ar_data if l=="ham")}')

# Combine
df = pd.concat([df_en, df_ar], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n✅ Total dataset: {len(df)} messages')
print(df['label'].value_counts())
df.head()

## 4️⃣ Exploratory Data Analysis (EDA)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
df['msg_length'] = df['message'].apply(len)
print(f'\nAvg length (Ham):  {df[df["label"]=="ham"]["msg_length"].mean():.1f} chars')
print(f'Avg length (Spam): {df[df["label"]=="spam"]["msg_length"].mean():.1f} chars')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts = df['label'].value_counts()
axes[0].pie(label_counts, labels=['Ham','Spam'], autopct='%1.1f%%',
            colors=['#2196F3','#F44336'], startangle=90)
axes[0].set_title('Label Distribution', fontsize=13, fontweight='bold')
df[df['label']=='ham']['msg_length'].plot(kind='hist', ax=axes[1],
    alpha=0.6, color='#2196F3', label='Ham', bins=30)
df[df['label']=='spam']['msg_length'].plot(kind='hist', ax=axes[1],
    alpha=0.6, color='#F44336', label='Spam', bins=30)
axes[1].set_title('Message Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Length (characters)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5️⃣ Text Preprocessing (Arabic + English)

In [ ]:
EN_SW = set([
    'i','me','my','we','our','you','your','he','him','his','she','her','it','its',
    'they','them','their','what','which','who','this','that','these','those','am',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','a','an','the','and','but','if','or','as','of','at','by','for','with',
    'to','from','up','in','out','on','off','then','here','there','when','where',
    'how','all','both','more','most','no','not','only','so','than','too','very',
    'can','will','just','should','now'
])
AR_SW = set([
    'في','من','إلى','على','عن','مع','هذا','هذه','ذلك','التي','الذي','وهو','وهي',
    'كان','كانت','يكون','تكون','هو','هي','هم','نحن','أنت','أنا','لكن','أو',
    'ثم','حتى','إذا','قد','لم','لن','ما','لا','إن','أن','بعد','قبل','كل',
    'بين','حول','خلال','عند','منذ','لدى','هل','كيف','أين','متى','لماذا',
    'الى','عبر','غير','فقط','حيث','اذا','ايضا','بل'
])

def preprocess_text(text):
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # remove tashkeel
    text = re.sub(r'[أإآا]', 'ا', text)                # normalize alef
    text = re.sub(r'ة', 'ه', text)                     # normalize ta marbuta
    text = re.sub(r'ى', 'ي', text)                     # normalize alef maqsura
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'\d+', '', text)                     # remove numbers
    text = re.sub(r'[^\u0600-\u06FFa-z\s]', '', text)  # keep AR+EN only
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split()
              if w not in EN_SW and w not in AR_SW and len(w) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['message'].apply(preprocess_text)
print('✅ Preprocessing done!')
print('\nEnglish example:')
print(f'  Original: {df["message"].iloc[0]}')
print(f'  Cleaned : {df["clean_text"].iloc[0]}')
print('\nArabic example:')
ar_idx = df[df['message'].str.contains('مبروك', na=False)].index[0]
print(f'  Original: {df["message"].iloc[ar_idx]}')
print(f'  Cleaned : {df["clean_text"].iloc[ar_idx]}')

## 6️⃣ TF-IDF Vectorization + Train/Test Split

In [ ]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label_num'],
    test_size=0.2, random_state=42, stratify=df['label_num']
)
print(f'Training samples: {len(X_train)}')
print(f'Testing samples : {len(X_test)}')

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f'TF-IDF shape: {X_train_tfidf.shape}')
print('✅ Text converted to numbers!')

## 7️⃣ Train Classifier (Naive Bayes)

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
print('✅ Model trained successfully!')

## 8️⃣ Evaluate Model

In [ ]:
y_pred = model.predict(X_test_tfidf)
print('=' * 42)
print('      MODEL EVALUATION RESULTS')
print('=' * 42)
print(f'  Accuracy  : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'  Precision : {precision_score(y_test, y_pred)*100:.2f}%')
print(f'  Recall    : {recall_score(y_test, y_pred)*100:.2f}%')
print(f'  F1-Score  : {f1_score(y_test, y_pred)*100:.2f}%')
print('=' * 42)
print()
print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

## 9️⃣ Hybrid Prediction (Keyword + ML)

In [ ]:
def predict_message(text):
    """Hybrid: keyword check first, then ML model."""
    # Step 1: keyword check
    kw_spam, matched_kw = keyword_check(text)
    # Step 2: ML model
    cleaned   = preprocess_text(text)
    vectorized = tfidf.transform([cleaned])
    ml_pred   = model.predict(vectorized)[0]
    ml_proba  = model.predict_proba(vectorized)[0]
    ml_conf   = ml_proba[ml_pred] * 100
    # Final: spam if EITHER flags it
    final = 1 if (kw_spam or ml_pred == 1) else 0
    label = 'SPAM 🚨' if final == 1 else 'HAM  ✅'

    msg_short = text[:65]+'...' if len(text)>65 else text
    print(f'Message   : {msg_short}')
    print(f'Result    : {label}')
    print(f'ML Model  : {"SPAM" if ml_pred==1 else "HAM"} ({ml_conf:.1f}% confidence)')
    if matched_kw:
        print(f'Keyword   : ⚠️  Matched → "{matched_kw}"')
    if final == 1:
        print('⚠️  WARNING  : Do NOT click links or share personal info!')
        print('⚠️  تحذير    : لا تضغط على أي روابط ولا تشارك بياناتك الشخصية!')
    print('-' * 65)

tests = [
    'WINNER!! You have been selected to receive $1000. Call now!',
    'Hey, are you coming to the team meeting at 3pm today?',
    'مبروك! لقد فزت بجائزة كبيرة. أرسل بياناتك الآن للمطالبة بجائزتك!',
    'هل ستحضر الاجتماع غداً؟ أرسل لي التقرير من فضلك.',
    'عاجل: تم اختراق حسابك البنكي. تحقق من بياناتك فوراً!',
    'واتساب: حسابك سيُحذف خلال 24 ساعة. انقر الرابط لتفعيله.',
    'كيف كانت إجازتك؟ أتمنى أن تكون استمتعت بها كثيراً.',
    'Congratulations! You won a free iPhone. Click here to claim!',
]
print('=' * 65)
print('  HYBRID SPAM DETECTOR — EN 🇬🇧 + AR 🇸🇦')
print('=' * 65)
for msg in tests:
    predict_message(msg)

## 🔟 Try Your Own Message

In [ ]:
# ✏️ اكتب رسالتك هنا بالعربي أو الإنجليزي
my_message = "اكتب رسالتك هنا للاختبار"
predict_message(my_message)

---
## ✅ Summary

| Step | Description |
|------|-------------|
| Dataset | SMS Spam Collection (5,574 EN) + Arabic (314 messages) |
| Arabic Categories | عروض وهمية + نصب + بنكية + واتساب/تلجرام |
| Languages | English 🇬🇧 + Arabic 🇸🇦 |
| Detection | Hybrid: Keyword-based + ML Model |
| Preprocessing | Normalize Arabic, remove diacritics, stopwords (EN+AR) |
| Vectorization | TF-IDF (5000 features, unigrams + bigrams) |
| Classifier | Multinomial Naive Bayes |
| Train/Test Split | 80% / 20% |
| Evaluation | Accuracy, Precision, Recall, F1-Score + Confusion Matrix |
| Bonus ✅ | Multiple Languages + Warning Messages + Streamlit UI |